# Procurement KPI Analysis: Supplier Performance DashboardReal-world procurement dataset (777 purchase orders, 5 suppliers, 2022-2023) sourced from Kaggle. This notebook builds an ETL pipeline to clean the data and analyze supplier performance across defect rate, delivery compliance, lead time, and cost savings.

In [ ]:
import pandas as pdimport numpy as npimport sqlite3import matplotlib.pyplot as pltimport seaborn as snsdf = pd.read_csv('procurement_data.csv')print(f"Loaded {len(df)} purchase order records.")df.head()

## Step 1: Data CleaningCheck for missing values and understand why they occur (e.g. cancelled orders have no delivery date).

In [ ]:
print("Missing values per column:")print(df.isnull().sum())print()print("Order status breakdown:")print(df['Order_Status'].value_counts())

In [ ]:
# Missing Delivery_Date and Defective_Units mostly occur for Cancelled/Pending orders# where no delivery happened yet - this is expected, not an errorprint(df[df['Delivery_Date'].isnull()]['Order_Status'].value_counts())print()print(df[df['Defective_Units'].isnull()]['Order_Status'].value_counts())

In [ ]:
# Convert date columnsdf['Order_Date'] = pd.to_datetime(df['Order_Date'])df['Delivery_Date'] = pd.to_datetime(df['Delivery_Date'])# Load into SQLite for SQL-based aggregationconn = sqlite3.connect(':memory:')df.to_sql('purchase_orders', conn, index=False, if_exists='replace')print("Data loaded into SQLite for SQL querying.")

## Step 2: Defect Rate by SupplierDefect rate only makes sense for orders that were actually delivered (fully or partially).

In [ ]:
query = '''SELECT Supplier,       COUNT(*) as delivered_orders,       AVG(CAST(Defective_Units AS FLOAT) / Quantity) * 100 as avg_defect_rate_pctFROM purchase_ordersWHERE Order_Status IN ('Delivered', 'Partially Delivered')  AND Defective_Units IS NOT NULLGROUP BY SupplierORDER BY avg_defect_rate_pct DESC'''defect_by_supplier = pd.read_sql(query, conn)print(defect_by_supplier)

In [ ]:
plt.figure(figsize=(9,5))sns.barplot(data=defect_by_supplier, x='avg_defect_rate_pct', y='Supplier', palette='Reds_r')plt.title('Average Defect Rate by Supplier', fontsize=13, fontweight='bold')plt.xlabel('Defect Rate (%)')plt.tight_layout()plt.savefig('defect_rate_by_supplier.png', dpi=120)plt.show()

## Step 3: Compliance Rate by Supplier

In [ ]:
query = '''SELECT Supplier,       COUNT(*) as total_orders,       SUM(CASE WHEN Compliance = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*) as compliance_rate_pctFROM purchase_ordersGROUP BY SupplierORDER BY compliance_rate_pct DESC'''compliance_by_supplier = pd.read_sql(query, conn)print(compliance_by_supplier)

## Step 4: Cost Savings from Price Negotiation

In [ ]:
df['Savings_per_unit'] = df['Unit_Price'] - df['Negotiated_Price']df['Total_Savings'] = df['Savings_per_unit'] * df['Quantity']savings_by_supplier = df.groupby('Supplier')['Total_Savings'].sum().sort_values(ascending=False)print("Total negotiation savings by supplier (INR):")print(savings_by_supplier)print()print(f"Total savings across all suppliers: Rs {df['Total_Savings'].sum():,.0f}")print(f"Total negotiated order value: Rs {(df['Negotiated_Price']*df['Quantity']).sum():,.0f}")

## Step 5: Delivery Lead Time by Supplier

In [ ]:
delivered_only = df[df['Order_Status']=='Delivered'].copy()delivered_only['Lead_Time_Days'] = (delivered_only['Delivery_Date'] - delivered_only['Order_Date']).dt.dayslead_time_by_supplier = delivered_only.groupby('Supplier')['Lead_Time_Days'].mean().sort_values()print("Average lead time by supplier (days):")print(lead_time_by_supplier)

## Step 6: Combined Supplier Risk ViewThe key finding: which supplier scores worst across multiple KPIs simultaneously?

In [ ]:
combined = defect_by_supplier.merge(compliance_by_supplier, on='Supplier').merge(    lead_time_by_supplier.reset_index(), on='Supplier').merge(savings_by_supplier.reset_index(), on='Supplier')combined = combined[['Supplier','avg_defect_rate_pct','compliance_rate_pct','Lead_Time_Days','Total_Savings']]combined.columns = ['Supplier','Defect_Rate_%','Compliance_%','Avg_Lead_Time_Days','Total_Savings_INR']print(combined.sort_values('Defect_Rate_%', ascending=False).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,5))sns.barplot(data=combined.sort_values('Defect_Rate_%', ascending=False),             x='Defect_Rate_%', y='Supplier', ax=axes[0], palette='Reds_r')axes[0].set_title('Defect Rate by Supplier (%)', fontweight='bold')sns.barplot(data=combined.sort_values('Compliance_%'),             x='Compliance_%', y='Supplier', ax=axes[1], palette='Greens')axes[1].set_title('Compliance Rate by Supplier (%)', fontweight='bold')plt.tight_layout()plt.savefig('supplier_kpi_dashboard.png', dpi=120)plt.show()

## Key Findings1. **Delta_Logistics** has both the highest defect rate (14.6%) and lowest compliance rate (60.8%) among all 5 suppliers — flagging it as the primary supplier risk in this dataset.2. **Epsilon_Group** is the strongest performer on quality and compliance (3.1% defect rate, 98.2% compliance), despite not offering the largest negotiated discount.3. Negotiation savings are fairly consistent across suppliers (~7.8%-8.2% average discount), suggesting defect rate and compliance — not price — are the real differentiators between suppliers.4. Lead time differences across suppliers are small (~9.9 to 11.2 days average), so delivery speed is not a major distinguishing factor here.**Conclusion:** Based on this analysis, Delta_Logistics should be flagged for a quality and compliance review, while Epsilon_Group represents the benchmark for supplier reliability in this dataset.